# Семинар 3. Цикл, бюджеты и трасса

**Python 3.10+ · только стандартная библиотека**

Это автономный учебный ноутбук: загрузите файл в Colab либо Jupyter и выполняйте ячейки сверху вниз.
Установка пакетов, ключи, GPU и сетевой доступ не требуются. Здесь используется **офлайн-симуляция**, а не настоящая LLM.

Предпосылки: функции, словари, исключения и базовые тесты `assert`; материал предыдущих занятий.
Четыре ячейки с тегом `exercise` содержат задания. После заполнения каждого задания запускайте его открытые проверки.
Пока функция не реализована, её вызов завершится `NotImplementedError`.

Баллы: 2 за каждое задание, 2 за корректный эксперимент и разбор ограничений.

Критерий сдачи: четыре функции, пройденные проверки, один дополнительный отрицательный случай и ответы в итоговом отчёте.
Результаты симуляции нельзя подписывать как качество или скорость языковой модели.


Реализуем последовательный harness с двумя видами действий: tool и finish.
    Политика scripted: она выдаёт заранее заданные словари. Часы виртуальные, стоимости инструментов условные.
    Это проверка жизненного цикла, **не live-LLM и не реализация принудительного прерывания процесса**.
    Таймаут обнаруживается до выбора действия и после исполнения; уже начавшийся инструмент может завершиться с эффектом.

    Источник для обсуждения замкнутого цикла: [ReAct](https://arxiv.org/abs/2210.03629).
    Таймауты Python изучаются отдельно по [официальной документации](https://docs.python.org/3/library/asyncio-task.html#asyncio.wait_for).


In [29]:
import copy
import json

def expect_error(exception, thunk):
    try:
        thunk()
    except exception:
        return
    raise AssertionError(f"Expected {exception.__name__}")

class FakeClock:
    def __init__(self): self.now = 0.0
    def advance(self, seconds): self.now += seconds

class ScriptedPolicy:
    def __init__(self, actions, repeat_last=False):
        self.actions = copy.deepcopy(actions)
        self.index = 0
        self.repeat_last = repeat_last
    def __call__(self, observations):
        if self.index >= len(self.actions):
            if not self.repeat_last:
                raise RuntimeError("script exhausted")
            return copy.deepcopy(self.actions[-1])
        action = self.actions[self.index]
        self.index += 1
        return copy.deepcopy(action)

class SimulatedExecutor:
    def __init__(self):
        self.clock = FakeClock()
        self.calls = []
        self.specs = {"lookup": {"cost": 1, "duration": 2}, "slow": {"cost": 2, "duration": 20}}
    def execute(self, name, args):
        self.calls.append({"name": name, "args": copy.deepcopy(args)})
        self.clock.advance(self.specs[name]["duration"])
        if set(args) != {"key"} or args["key"] != "room":
            return {"ok": False, "error": "not_found"}
        return {"ok": True, "value": "314"}

def verify_answer(answer, observations):
    return answer == "314" and any(o.get("ok") and o.get("value") == "314" for o in observations)

LOOKUP = {"kind": "tool", "name": "lookup", "args": {"key": "room"}}
FINISH = {"kind": "finish", "answer": "314"}
print("OFFLINE: scripted policy + virtual time")


OFFLINE: scripted policy + virtual time


### Решённый пример: полезное наблюдение после срока
    Сам факт истечения срока не отменяет выполненное чтение. Часы двигаются детерминированно.


In [30]:
example = SimulatedExecutor()
deadline = example.clock.now + 1
observation = example.execute("lookup", {"key": "room"})
assert observation["ok"] and example.clock.now > deadline
print({"observation": observation, "finished_at": example.clock.now, "deadline": deadline, "within_deadline": False})


{'observation': {'ok': True, 'value': '314'}, 'finished_at': 2.0, 'deadline': 1.0, 'within_deadline': False}


## Задание 1. Язык действий — 2 балла

`validate_action(action)` возвращает независимую копию допустимого действия, иначе вызывает ValueError.
    Разрешены ровно две формы: tool с ключами kind, name, args (name — непустая строка, args — dict)
    и finish с ключами kind, answer (answer — строка). kind строго tool/finish, лишние поля запрещены.
    Это проверка формы; наличие инструмента в реестре проверяется позднее.


In [31]:
def validate_action(action):
    action1 = copy.deepcopy(action)
    if "kind"  not in action:
        raise ValueError
    elif action["kind"] == "tool":
        if "name" not in action or "args" not in action:
            raise ValueError
        else:
            if type(action["name"]) != str or action["name"] == "":
                raise ValueError
            if type(action["args"]) != dict:
                raise ValueError
            if len(action) != 3:
                raise ValueError
            
    elif action["kind"] == "finish":
        if "answer" not in action:
            raise ValueError
        else:
            if type(action["answer"]) != str:
                raise ValueError
            if len(action) != 2:
                raise ValueError
    else:
        raise ValueError
    return action1
             



    
    raise NotImplementedError("Student exercise")


In [32]:
assert validate_action(LOOKUP) == LOOKUP
assert validate_action(FINISH) == FINISH
for bad in ([], {"kind": "other"}, {"kind": "finish", "answer": 7}, {**LOOKUP, "extra": 1}, {"kind": "tool", "name": "", "args": {}}):
    expect_error(ValueError, lambda bad=bad: validate_action(bad))
saved = validate_action(LOOKUP); saved["args"]["key"] = "changed"
assert LOOKUP["args"]["key"] == "room"
print("PASS: validate_action")


PASS: validate_action


## Задание 2. Бюджет до исполнения — 2 балла

`admit_request(calls, spent, next_cost, max_calls, max_cost)` возвращает None при допуске,
    иначе call_budget или cost_budget. Число calls — уже исполненные вызовы; новый не допускается при calls >= max_calls.
    Стоимость допускается при spent + next_cost <= max_cost. При нарушении обоих лимитов приоритет call_budget.
    Предусловие: все числа неотрицательные, стоимость заранее известна; функция ничего не меняет.


In [37]:
def admit_request(calls, spent, next_cost, max_calls, max_cost):
    if calls >= max_calls:
        return "call_budget"
    if spent + next_cost > max_cost:
        return "cost_budget"
    return None

    
    raise NotImplementedError("Student exercise")


In [38]:
assert admit_request(0, 0, 1, 1, 1) is None
assert admit_request(1, 1, 1, 1, 1) == "call_budget"
assert admit_request(0, 1, 2, 3, 2) == "cost_budget"
assert admit_request(0, 3, 2, 1, 5) is None
assert admit_request(0, 0, 0, 0, 0) == "call_budget"
print("PASS: admit_request")


PASS: admit_request


## Задание 3. Независимая безопасная запись — 2 балла

`make_event(index, kind, status, payload, now)` возвращает dict с этими пятью ключами.
    payload содержит только JSON-совместимые структуры. Рекурсивно замените значения полей api_key, token, password
    (без учёта регистра) строкой <redacted>; обойдите вложенные списки/словари.
    Исходные данные не меняйте; событие не должно ссылаться на изменяемые вложенные контейнеры payload.
    Это узкое правило скрытия известных полей, а не универсальный детектор секретов в свободном тексте.


In [ ]:
import copy

def dict_walk(d):
    if d is None:
        return
    if type(d) == dict:
        for keys in d:
            if isinstance(keys, str) and (keys.lower() == "api_key" or keys.lower() == "password" or keys.lower() == "token"):
                
                d[keys] = "<redacted>"
          
            elif type(d[keys]) == dict or type(d[keys]) == list:
                dict_walk(d[keys])
                
  
    elif type(d) == list:
        for item in d:
            dict_walk(item)

def make_event(index, kind, status, payload, now):
    result = dict()
    result["index"] = index
    result["kind"] = kind
    result["status"] = status
    
    result["payload"] = copy.deepcopy(payload)
    result["now"] = now
    dict_walk(result["payload"])
    
    return result

In [57]:
payload = {"args": {"token": "secret", "x": [1, {"PASSWORD": "hidden"}]}}
event = make_event(0, "tool", "ok", payload, 2)
print(event["payload"]["args"]["token"])
assert event["payload"]["args"]["token"] == "<redacted>"
assert event["payload"]["args"]["x"][1]["PASSWORD"] == "<redacted>"
payload["args"]["x"][0] = 9
assert event["payload"]["args"]["x"][0] == 1
assert payload["args"]["token"] == "secret"
json.dumps(event)
print("PASS: make_event")


<redacted>
PASS: make_event


## Задание 4. Ограниченный цикл — 2 балла

Реализуйте `run_loop` по следующему контракту.
    На каждом из max_turns решений: если now >= deadline, остановитесь с timeout; получите и проверьте действие;
    запишите decision со статусом proposed. Ошибка формы: invalid_action; исключение политики: policy_error.
    finish проверяется `verify(answer, observations)` и даёт success/rejected.
    Для tool: проверьте имя, затем бюджет; после допуска **до execute** увеличьте calls и spent.
    Сохраните наблюдение и tool-событие (ok/error). После вызова снова проверьте срок. Ошибка инструмента не запускает скрытый retry.
    По исчерпании turns — turn_budget. Каждый исход завершайте одним stop-событием.
    Возвращайте {status, answer, observations, trace, calls, spent}; answer задан только для finish, иначе None.
    Параметры бюджета здесь неотрицательные, max_turns и max_calls целые; execute и verify — доверенная инфраструктура.
    Исключение execute преобразуйте в {ok: False, error: tool_exception}, не раскрывая текст.


In [ ]:
def run_loop(policy, executor, verify, max_turns=5, max_calls=3, max_cost=5, timeout=10):
    # TODO: реализуйте контракт из предыдущей ячейки.
    raise NotImplementedError("Student exercise")


In [ ]:
executor = SimulatedExecutor()
result = run_loop(ScriptedPolicy([LOOKUP, FINISH]), executor, verify_answer, max_calls=1, max_cost=1)
assert result["status"] == "success" and result["calls"] == 1 and result["spent"] == 1
assert [e["index"] for e in result["trace"]] == list(range(len(result["trace"])))
assert result["trace"][-1]["kind"] == "stop"
assert sum(e["kind"] == "stop" for e in result["trace"]) == 1
assert run_loop(ScriptedPolicy([FINISH]), SimulatedExecutor(), verify_answer)["status"] == "rejected"
blocked = SimulatedExecutor()
assert run_loop(ScriptedPolicy([LOOKUP]), blocked, verify_answer, max_cost=0)["status"] == "cost_budget"
assert blocked.calls == []
looped = run_loop(ScriptedPolicy([LOOKUP], repeat_last=True), SimulatedExecutor(), verify_answer, max_calls=1)
assert looped["status"] == "call_budget" and looped["calls"] == 1
timed = run_loop(ScriptedPolicy([LOOKUP, FINISH]), SimulatedExecutor(), verify_answer, timeout=1)
assert timed["status"] == "timeout" and timed["calls"] == 1 and timed["observations"][0]["ok"]
assert run_loop(ScriptedPolicy([LOOKUP]), SimulatedExecutor(), verify_answer, max_turns=1)["status"] == "turn_budget"
assert run_loop(ScriptedPolicy([{"kind": "oops"}]), SimulatedExecutor(), verify_answer)["status"] == "invalid_action"
print("PASS: run_loop")


## Эксперимент
    До запуска заполните прогноз: статус, calls и spent для каждого бюджета. Время условное, не wall-clock.
    Сравните число событий с числом инструментов: decision и stop не являются инструментальными вызовами.
    Затем замените первое действие на slow и объясните, почему просрочка не стирает наблюдение.


In [ ]:
rows = []
for limit in [0, 1, 2]:
    result = run_loop(ScriptedPolicy([LOOKUP, FINISH]), SimulatedExecutor(), verify_answer, max_calls=limit)
    rows.append({"max_calls": limit, "status": result["status"], "calls": result["calls"], "spent": result["spent"]})
print(json.dumps(rows, ensure_ascii=False, indent=2))
assert [r["status"] for r in rows] == ["call_budget", "success", "success"]
slow = {"kind": "tool", "name": "slow", "args": {"key": "room"}}
slow_result = run_loop(ScriptedPolicy([slow, FINISH]), SimulatedExecutor(), verify_answer)
assert slow_result["status"] == "timeout" and slow_result["observations"][0]["value"] == "314"
print(json.dumps(slow_result["trace"], ensure_ascii=False, indent=2))


## Отчёт
    1. Где проверяется бюджет до запуска и какое наблюдение подтверждает отсутствие запрещённого вызова?
    2. Почему finish после одного чтения не нарушает max_calls=1?
    3. Чем simulated timeout отличается от прерывания настоящего процесса?
    4. Добавьте проверку ошибки execute: вызов должен быть учтён, исключение не раскрывается, наблюдение содержит tool_exception.
    5. Назовите два инварианта трассы и покажите соответствующие значения.

    Дополнительно: спроектируйте retry только для читающего инструмента, включая его бюджет и трассу; не вводите скрытых повторов.
